In [ ]:
# Librerías importantes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Quitar mensajes de advertencia
import warnings
warnings.filterwarnings("ignore")

# Configuración estética
sns.set_theme(style="whitegrid")
df = pd.read_csv(r"C:\Users\danih\OneDrive\Documentos\U\6° SEMESTRE\Machine L\archive (5)\diabetes_binary_health_indicators_BRFSS2015.csv", index_col=None)

#Colores a utilizar
coral = "#E2A8A4"
aqua = "#56D8CB"
colors = [aqua, coral]

## **Correr svm**

In [ ]:
#Organización de variables para modelado

#NOTA:
#Para la etapa de modelado se trabajará con `df` como base principal, ya que `df_viz` fue construido con fines de visualización y varias variables fueron recodificadas como categóricas para facilitar el EDA.
# Así: `df`  -> se usará para preprocesamiento y modelado
#`df_viz` -> se conserva únicamente para visualizaciones y análisis gráfico

df_model = df.copy()

#Variable objetivo ----------------------------------------------
#Ya habíamos definido la variable objetivo como "Diabetes_binary" en etapas anteriores, pero se reafirma aquí para mantener claridad y estructura en esta sección de modelado.
#Objetivo = "Diabetes_binary"

#Definición de grupos de variables -----------------------------
#Se clasifican las variables predictoras en grupos según su tipo, lo cual facilitará la selección de técnicas de preprocesamiento y modelado específicas para cada tipo de variable.
#Ya también se habían identificado estas variables en etapas anteriores.
#binary_feature_cols 
#numeric_cols
#ordinal_cols 

 
#En este dataset en particular no quedan variables nominales puras, porque la mayoría vienen ya codificadas como enteros. Se deja la lista definida para mantener una estructura reusable.
#categorical_cols = []


#Verificación básica----------------------------------------------

all_defined_cols = numeric_cols + ordinal_cols + binary_feature_cols + [Objetivo]

predictor_cols = [col for col in df_model.columns if col != Objetivo]

missing_in_definition = sorted(set(predictor_cols) - set(all_defined_cols))
duplicated_in_groups = sorted([col for col in all_defined_cols if all_defined_cols.count(col) > 1])
#missing_in_definition
#duplicated_in_groups

print("Variable objetivo:", Objetivo)
print("\nNúmero total de predictores en df_model:", len(predictor_cols))
print("Número total de variables clasificadas:", len(all_defined_cols))

print("\nResumen por grupo:")
print(f"- Numéricas continuas: {len(numeric_cols)}")
print(f"- Ordinales: {len(ordinal_cols)}")
print(f"- Binarias: {len(binary_feature_cols)}")

print("\nVariables numéricas continuas:", numeric_cols)
print("Variables ordinales:", ordinal_cols)
print("Variables binarias:", binary_feature_cols)


#Separación X e y---------------------------------------------------------------------------

X = df_model.drop(columns=[Objetivo])
y = df_model[Objetivo]

print("\nShape de X:", X.shape)
print("Shape de y:", y.shape)

print("\nDistribución de la variable objetivo:")
print(y.value_counts().sort_index())
print("\nProporción de la variable objetivo:")
print(y.value_counts(normalize=True).sort_index())

In [ ]:


from operator import index

from sklearn.model_selection import train_test_split, StratifiedKFold

#Partición train/test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

split_summary = pd.DataFrame({
    "Conjunto": ["Train", "Test"],
    "Observaciones": [len(X_train), len(X_test)],
    "Porcentaje": [
        f"{len(X_train) / len(df_model) * 100:.1f}%",
        f"{len(X_test) / len(df_model) * 100:.1f}%"
    ],
    "Shape X": [str(X_train.shape), str(X_test.shape)],
    "Shape y": [str(y_train.shape), str(y_test.shape)]
})

display(split_summary.style.hide(axis="index"))

#Verificación de proporciones de clases

train_dist = y_train.value_counts(normalize=True).sort_index() * 100
test_dist = y_test.value_counts(normalize=True).sort_index() * 100

distribution_df = pd.DataFrame({
    "Clase": ["Sin diabetes", "Prediabetes/Diabetes"],
    "Train (%)": train_dist.values,
    "Test (%)": test_dist.values
})

display(distribution_df)

In [ ]:
from sklearn.model_selection import StratifiedKFold

#Esquema de validación cruzada
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv_strategy)

#Verificación de la proporción de clases en cada fold

overall_dist = y_train.value_counts(normalize=True).sort_index() * 100

fold_summary = []

for fold, (train_idx, val_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    train_dist = y_fold_train.value_counts(normalize=True).sort_index() * 100
    val_dist = y_fold_val.value_counts(normalize=True).sort_index() * 100

    fold_summary.append({
        "Fold": fold,
        "Train n": len(train_idx),
        "Val n": len(val_idx),
        "Train clase 0 (%)": round(train_dist.get(0, 0), 2),
        "Train clase 1 (%)": round(train_dist.get(1, 0), 2),
        "Val clase 0 (%)": round(val_dist.get(0, 0), 2),
        "Val clase 1 (%)": round(val_dist.get(1, 0), 2),
        "Global clase 0 (%)": round(overall_dist.get(0, 0), 2),
        "Global clase 1 (%)": round(overall_dist.get(1, 0), 2)
    })

fold_summary_df = pd.DataFrame(fold_summary)
display(fold_summary_df)

In [ ]:
#Función auxiliar para construir el preprocesador

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def build_preprocessor(scale_numeric=True):
    """
    Construye un preprocesador según el tipo de variable.
    
    Parámetros
    ----------
    scale_numeric : bool
        Si True, escala variables numéricas continuas.
        Si False, solo imputa.
    """

    if scale_numeric:
        numeric_transformer_local = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_transformer_local = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])

    ordinal_transformer_local = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    binary_transformer_local = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor_local = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_local, numeric_cols),
            ("ord", ordinal_transformer_local, ordinal_cols),
            ("bin", binary_transformer_local, binary_feature_cols)
        ],
        remainder="drop"
    )

    return preprocessor_local

In [ ]:
import pandas as pd
import numpy as np

#Construir preprocesador de inspección
preprocessor = build_preprocessor(scale_numeric=True)

print(preprocessor)

#Aplicar preprocesamiento únicamente sobre entrenamiento
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)



#Obtener nombres finales de columnas
final_feature_names = numeric_cols + ordinal_cols + binary_feature_cols

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=final_feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=final_feature_names
)

#Comparación de shapes
shape_summary = pd.DataFrame({
    "Conjunto": [
        "X_train original",
        "X_train transformado",
        "X_test original",
        "X_test transformado"
    ],
    "Shape": [
        str(X_train.shape),
        str(X_train_processed_df.shape),
        str(X_test.shape),
        str(X_test_processed_df.shape)
    ]
})

display(shape_summary)

#Verificación de escalado en variables numéricas
scaling_check = pd.DataFrame({
    "Variable": numeric_cols,
    "Media después de escalado": [
        X_train_processed_df[col].mean() for col in numeric_cols
    ],
    "Desv. estándar después de escalado": [
        X_train_processed_df[col].std() for col in numeric_cols
    ]
})

display(scaling_check.round(4))

In [ ]:
from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp, chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

SAMPLE_FRAC = 0.30

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=SAMPLE_FRAC,
    stratify=y_train,
    random_state=42
)

print("Tamaño X_train completo:", X_train.shape)
print("Tamaño muestra estratificada:", X_train_sample.shape)
print(f"Proporción utilizada: {SAMPLE_FRAC * 100:.0f}% del conjunto de entrenamiento")


# Comparación de distribución de clases y ratio de desbalance

def resumen_clases(y, nombre):
    conteo = y.value_counts().sort_index()
    prop = y.value_counts(normalize=True).sort_index() * 100
    ratio = conteo.iloc[0] / conteo.iloc[1]

    return {
        "Conjunto": nombre,
        "n total": len(y),
        "n clase 0": conteo.iloc[0],
        "n clase 1": conteo.iloc[1],
        "Clase 0 (%)": prop.iloc[0],
        "Clase 1 (%)": prop.iloc[1],
        "Ratio 0:1": ratio
    }

class_comparison = pd.DataFrame([
    resumen_clases(y_train, "Train completo"),
    resumen_clases(y_train_sample, "Muestra estratificada"),
    resumen_clases(y_test, "Test")
])

display(class_comparison.round(4))

# Diferencias entre train completo y muestra
diff_c0 = abs(class_comparison.loc[0, "Clase 0 (%)"] - class_comparison.loc[1, "Clase 0 (%)"])
diff_c1 = abs(class_comparison.loc[0, "Clase 1 (%)"] - class_comparison.loc[1, "Clase 1 (%)"])
diff_ratio = abs(class_comparison.loc[0, "Ratio 0:1"] - class_comparison.loc[1, "Ratio 0:1"])

print(f"Diferencia clase 0 entre train y muestra: {diff_c0:.6f}%")
print(f"Diferencia clase 1 entre train y muestra: {diff_c1:.6f}%")
print(f"Diferencia ratio 0:1 entre train y muestra: {diff_ratio:.6f}")

# Comparación de estadísticos descriptivos

desc_full = X_train[numeric_cols + ordinal_cols].describe().T[["mean", "std", "50%"]]
desc_sample = X_train_sample[numeric_cols + ordinal_cols].describe().T[["mean", "std", "50%"]]

representativity_table = desc_full.join(
    desc_sample,
    lsuffix="_train",
    rsuffix="_sample"
)

representativity_table["diff_mean_abs"] = (
    representativity_table["mean_train"] - representativity_table["mean_sample"]
).abs()

representativity_table["diff_median_abs"] = (
    representativity_table["50%_train"] - representativity_table["50%_sample"]
).abs()

display(representativity_table.round(4))


# Pruebas KS para variables numéricas/ordinales

ks_results = []

for col in numeric_cols + ordinal_cols:
    stat, p_value = ks_2samp(X_train[col], X_train_sample[col])
    ks_results.append({
        "Variable": col,
        "KS statistic": stat,
        "p-value": p_value
    })

ks_results_df = pd.DataFrame(ks_results)
display(ks_results_df.round(4))


# Pruebas chi-cuadrado para variables binarias

chi_results = []

for col in binary_feature_cols:
    
    full_values = X_train[col].reset_index(drop=True)
    sample_values = X_train_sample[col].reset_index(drop=True)
    
    group_labels = pd.Series(
        ["Train completo"] * len(full_values) + ["Muestra"] * len(sample_values)
    )
    
    values = pd.concat(
        [full_values, sample_values],
        ignore_index=True
    )
    
    table = pd.crosstab(group_labels, values)
    
    chi2, p_value, dof, expected = chi2_contingency(table)

    chi_results.append({
        "Variable": col,
        "Chi2": chi2,
        "p-value": p_value
    })

chi_results_df = pd.DataFrame(chi_results)
display(chi_results_df.round(4))


In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


def run_benchmark_classifier(
    model_name,
    estimator,
    scale_numeric=True,
    cv=None,
    n_jobs=-1,
    X_train_data=None,
    y_train_data=None,
    X_test_data=None,
    y_test_data=None
):
    """
    Ejecuta un benchmark de clasificación con hiperparámetros fijos.
    """

    if cv is None:
        cv = cv_strategy

    if X_train_data is None:
        X_train_data = X_train
    if y_train_data is None:
        y_train_data = y_train
    if X_test_data is None:
        X_test_data = X_test
    if y_test_data is None:
        y_test_data = y_test

    preprocessor_local = build_preprocessor(scale_numeric=scale_numeric)

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_local),
        ("model", estimator)
    ])

    scoring_metrics = {
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "auc": "roc_auc"
    }

    start_time = time.time()

    cv_results = cross_validate(
        estimator=pipe,
        X=X_train_data,
        y=y_train_data,
        cv=cv,
        scoring=scoring_metrics,
        return_train_score=False,
        n_jobs=n_jobs
    )

    pipe.fit(X_train_data, y_train_data)

    elapsed_time = time.time() - start_time

    y_pred = pipe.predict(X_test_data)

    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_test_data)[:, 1]
    elif hasattr(pipe, "decision_function"):
        y_score = pipe.decision_function(X_test_data)
    else:
        y_score = None

    results = {
        "Modelo": model_name,
        "CV Accuracy": np.mean(cv_results["test_accuracy"]),
        "CV Precision": np.mean(cv_results["test_precision"]),
        "CV Recall": np.mean(cv_results["test_recall"]),
        "CV F1": np.mean(cv_results["test_f1"]),
        "CV AUC": np.mean(cv_results["test_auc"]),
        "Test Accuracy": accuracy_score(y_test_data, y_pred),
        "Test Precision": precision_score(y_test_data, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test_data, y_pred, zero_division=0),
        "Test F1": f1_score(y_test_data, y_pred, zero_division=0),
        "Test AUC": roc_auc_score(y_test_data, y_score) if y_score is not None else np.nan,
        "Tiempo (s)": elapsed_time,
        "Predicciones": y_pred,
        "Scores": y_score,
        "Pipeline": pipe,
        "Matriz Confusión": confusion_matrix(y_test_data, y_pred),
        "Reporte": classification_report(y_test_data, y_pred, output_dict=True)
    }

    return results


In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, ADASYN

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

RANDOM_STATE = 42
PRIMARY_SCORING = "recall"

imbalance_ratio = (
    y_train.value_counts().sort_index().iloc[0] /
    y_train.value_counts().sort_index().iloc[1]
)

balance_strategies = ["none", "smote", "adasyn", "class_weight"]

In [ ]:
classification_models = {
    "knn": {
        "name": "KNN",
        "scale_numeric": True,
        "sample": False
    },
    "naive_bayes": {
        "name": "Gaussian Naive Bayes",
        "scale_numeric": True,
        "sample": False
    },
    "logistic_l1": {
        "name": "Logistic Regression L1",
        "scale_numeric": True,
        "sample": False
    },
    "logistic_l2": {
        "name": "Logistic Regression L2",
        "scale_numeric": True,
        "sample": False
    },
    "decision_tree": {
        "name": "Decision Tree",
        "scale_numeric": False,
        "sample": False
    },
    "random_forest": {
        "name": "Random Forest",
        "scale_numeric": False,
        "sample": False
    },
    "xgboost": {
        "name": "XGBoost",
        "scale_numeric": False,
        "sample": False
    },
    "svm": {
        "name": "SVM RBF",
        "scale_numeric": True,
        "sample": True
    }
}

In [ ]:
def get_train_data(model_key):
    if classification_models[model_key]["sample"]:
        return X_train_sample, y_train_sample
    return X_train, y_train


def supports_class_weight(model_key):
    return model_key in [
        "logistic_l1",
        "logistic_l2",
        "decision_tree",
        "random_forest",
        "svm"
    ]
    
def get_balance_options(model_key):
    if model_key == "xgboost":
        return ["none", "smote", "adasyn", "scale_pos_weight"]

    options = ["none", "smote", "adasyn"]

    if supports_class_weight(model_key):
        options.append("class_weight")

    return options


def make_sampler_for_search(balance_strategy):
    if balance_strategy == "smote":
        return SMOTE(random_state=RANDOM_STATE)
    elif balance_strategy == "adasyn":
        return ADASYN(random_state=RANDOM_STATE)
    else:
        return "passthrough"


def make_estimator_for_search(model_key, balance_strategy="none"):
    """
    Define el modelo base y ajusta class_weight/scale_pos_weight cuando aplica.
    """
    
    class_weight_value = "balanced" if balance_strategy == "class_weight" else None

    if model_key == "knn":
        return KNeighborsClassifier()

    if model_key == "naive_bayes":
        return GaussianNB()

    if model_key == "logistic_l2":
        return LogisticRegression(
            penalty="l2",
            solver="liblinear",
            max_iter=1000,
            class_weight=class_weight_value,
            random_state=RANDOM_STATE
        )

    if model_key == "logistic_l1":
        return LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=1000,
            class_weight=class_weight_value,
            random_state=RANDOM_STATE
        )

    if model_key == "decision_tree":
        return DecisionTreeClassifier(
            class_weight=class_weight_value,
            random_state=RANDOM_STATE
        )

    if model_key == "random_forest":
        return RandomForestClassifier(
            class_weight=class_weight_value,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    if model_key == "xgboost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=imbalance_ratio if balance_strategy == "scale_pos_weight" else 1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    if model_key == "svm":
        return SVC(
            kernel="rbf",
            probability=True,
            class_weight=class_weight_value,
            random_state=RANDOM_STATE
        )


def make_pipeline_for_search(model_key, balance_strategy, scale_numeric=True):
    estimator = make_estimator_for_search(model_key, balance_strategy)
    sampler = make_sampler_for_search(balance_strategy)

    pipe = ImbPipeline(steps=[
        ("preprocessor", build_preprocessor(scale_numeric=scale_numeric)),
        ("sampler", sampler),
        ("model", estimator)
    ])

    return pipe

In [ ]:
param_spaces_grid = {
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"]
    },

    "naive_bayes": {
         "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-7, 1e-5, 1e-3]
    },

    "logistic_l2": {
        "model__C": [0.01, 0.1, 1, 10]
    },

    "logistic_l1": {
        "model__C": [0.01, 0.1, 1, 10]
    },

    "decision_tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [5, 10, 15, None],
        "model__min_samples_split": [2, 10, 30],
        "model__min_samples_leaf": [1, 5, 10]
    },

    "random_forest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [10, 15, 20, None],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4, 5],
        "model__max_features": ["sqrt", "log2"] 
    },

    "xgboost": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5],
        "model__learning_rate": [0.03, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__min_child_weight": [1, 3, 5]
    },

    "svm": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1]
    }
}

**GRID**

In [ ]:
def run_grid_search_model(model_key, balance_strategy):

    model_info = classification_models[model_key]
    X_train_used, y_train_used = get_train_data(model_key)

    pipe = make_pipeline_for_search(
        model_key=model_key,
        balance_strategy=balance_strategy,
        scale_numeric=model_info["scale_numeric"]
    )

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_spaces_grid[model_key],
        scoring=PRIMARY_SCORING,
        cv=cv_strategy,
        n_jobs=-1,
        refit=True
    )

    start_time = time.time()
    grid.fit(X_train_used, y_train_used)
    elapsed_time = time.time() - start_time

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)

    return {
        "Modelo": model_info["name"],
        "Metodo": "Grid Search",
        "Balanceo": balance_strategy,
        "CV Recall": grid.best_score_,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0),
        "Test AUC": roc_auc_score(y_test, y_score),
        "Tiempo (s)": elapsed_time,
        "Evaluaciones": len(grid.cv_results_["params"]),
        "Best Params": grid.best_params_,
        "Best Estimator": best_model,
        "Matriz Confusión": confusion_matrix(y_test, y_pred),
        "Predicciones": y_pred,
        "Scores": y_score
    }

**RANDOM SEARCH**

In [ ]:
param_distributions_random = {
    "random_forest": {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [10, 20, 30, None],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": ["sqrt", "log2"]
    },

    "svm": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.001, 0.01, 0.1, 1]
    },
    
    "decision_tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [5, 10, 15, 20, None],
        "model__min_samples_split": [2, 5, 10, 20, 30],
        "model__min_samples_leaf": [1, 2, 5, 10]
    },

    "logistic_l1": {
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
    },

    "logistic_l2": {
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
    }
}

In [ ]:
def run_random_search_model(model_key, balance_strategy, n_iter=30):

    model_info = classification_models[model_key]
    X_train_used, y_train_used = get_train_data(model_key)

    pipe = make_pipeline_for_search(
        model_key=model_key,
        balance_strategy=balance_strategy,
        scale_numeric=model_info["scale_numeric"]
    )

    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions_random[model_key],
        n_iter=n_iter,
        scoring=PRIMARY_SCORING,
        cv=cv_strategy,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    start_time = time.time()
    random_search.fit(X_train_used, y_train_used)
    elapsed_time = time.time() - start_time

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)

    return {
        "Modelo": model_info["name"],
        "Metodo": "Random Search",
        "Balanceo": balance_strategy,
        "CV Recall": random_search.best_score_,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0),
        "Test AUC": roc_auc_score(y_test, y_score),
        "Tiempo (s)": elapsed_time,
        "Evaluaciones": len(random_search.cv_results_["params"]),
        "Best Params": random_search.best_params_,
        "Best Estimator": best_model,
        "Matriz Confusión": confusion_matrix(y_test, y_pred),
        "Predicciones": y_pred,
        "Scores": y_score
    }

**OPTUNA**

In [ ]:
import optuna

def suggest_params_optuna(trial, model_key):

    if model_key == "knn":
        return {
            "n_neighbors": trial.suggest_int("n_neighbors", 3, 29, step=2),
            "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
            "metric": trial.suggest_categorical("metric", ["euclidean", "manhattan"])
        }

    if model_key == "naive_bayes":
        return {
            "var_smoothing": trial.suggest_float("var_smoothing", 1e-10, 1e-6, log=True)
        }

    if model_key in ["logistic_l1", "logistic_l2"]:
        return {
            "C": trial.suggest_float("C", 0.001, 10, log=True)
        }

    if model_key == "decision_tree":
        return {
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_depth": trial.suggest_categorical("max_depth", [5, 10, 15, None]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10)
        }

    if model_key == "random_forest":
        return {
            "n_estimators": trial.suggest_categorical("n_estimators", [100, 200, 300]),
            "max_depth": trial.suggest_categorical("max_depth", [5, 20, None]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]) 
        }

    if model_key == "xgboost":
        return {
            "n_estimators": trial.suggest_categorical("n_estimators", [100, 200, 300]),
            "max_depth": trial.suggest_int("max_depth", 3, 20),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 5)
        }

    if model_key == "svm":
        return {
            "C": trial.suggest_float("C", 0.01, 100, log=True),
            "gamma": trial.suggest_categorical("gamma", ["scale", 0.001, 1])
        }

In [ ]:
def make_estimator_from_params(model_key, params, balance_strategy):

    if model_key == "knn":
        return KNeighborsClassifier(**params)

    if model_key == "naive_bayes":
        return GaussianNB(**params)

    if model_key == "logistic_l1":
        return LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=1000,
            class_weight="balanced" if balance_strategy == "class_weight" else None,
            random_state=RANDOM_STATE,
            **params
        )

    if model_key == "logistic_l2":
        return LogisticRegression(
            penalty="l2",
            solver="liblinear",
            max_iter=1000,
            class_weight="balanced" if balance_strategy == "class_weight" else None,
            random_state=RANDOM_STATE,
            **params
        )

    if model_key == "decision_tree":
        return DecisionTreeClassifier(
            class_weight="balanced" if balance_strategy == "class_weight" else None,
            random_state=RANDOM_STATE,
            **params
        )

    if model_key == "random_forest":
        return RandomForestClassifier(
            class_weight="balanced" if balance_strategy == "class_weight" else None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_key == "xgboost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=imbalance_ratio if balance_strategy == "scale_pos_weight" else 1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_key == "svm":
        return SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced" if balance_strategy == "class_weight" else None,
            random_state=RANDOM_STATE,
            **params
        )

In [ ]:
def objective_optuna(trial, model_key, scale_numeric=True):

    X_model_train, y_model_train = get_train_data(model_key)

    available_balance = get_balance_options(model_key)

    balance_strategy = trial.suggest_categorical("balance_strategy", available_balance)
    params = suggest_params_optuna(trial, model_key)

    fold_scores = []

    for train_idx, val_idx in cv_strategy.split(X_model_train, y_model_train):

        X_fold_train = X_model_train.iloc[train_idx]
        X_fold_val = X_model_train.iloc[val_idx]

        y_fold_train = y_model_train.iloc[train_idx]
        y_fold_val = y_model_train.iloc[val_idx]

        preprocessor_fold = build_preprocessor(scale_numeric=scale_numeric)

        X_fold_train_processed = preprocessor_fold.fit_transform(X_fold_train)
        X_fold_val_processed = preprocessor_fold.transform(X_fold_val)

        if balance_strategy == "smote":
            sampler = SMOTE(random_state=RANDOM_STATE)
            X_fold_train_processed, y_fold_train = sampler.fit_resample(
                X_fold_train_processed,
                y_fold_train
            )

        elif balance_strategy == "adasyn":
            sampler = ADASYN(random_state=RANDOM_STATE)
            X_fold_train_processed, y_fold_train = sampler.fit_resample(
                X_fold_train_processed,
                y_fold_train
            )

        model = make_estimator_from_params(
            model_key=model_key,
            params=params,
            balance_strategy=balance_strategy
        )

        model.fit(X_fold_train_processed, y_fold_train)
        y_fold_pred = model.predict(X_fold_val_processed)

        fold_scores.append(
            recall_score(y_fold_val, y_fold_pred, zero_division=0)
        )

    return np.mean(fold_scores)

In [ ]:
def run_optuna_model(model_key, model_name=None, scale_numeric=None, n_trials=30):

    model_info = classification_models[model_key]

    if model_name is None:
        model_name = model_info["name"]

    if scale_numeric is None:
        scale_numeric = model_info["scale_numeric"]

    study = optuna.create_study(direction="maximize")

    start_time = time.time()

    study.optimize(
        lambda trial: objective_optuna(
            trial,
            model_key=model_key,
            scale_numeric=scale_numeric
        ),
        n_trials=n_trials
    )

    elapsed_time = time.time() - start_time

    best_params_all = study.best_params.copy()
    best_balance = best_params_all.pop("balance_strategy")

    best_model = make_pipeline_for_search(
        model_key=model_key,
        balance_strategy=best_balance,
        scale_numeric=scale_numeric
    )

    best_model.set_params(**{
        f"model__{k}": v for k, v in best_params_all.items()
    })

    X_train_used, y_train_used = get_train_data(model_key)

    best_model.fit(X_train_used, y_train_used)

    y_pred = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)

    return {
        "Modelo": model_name,
        "Metodo": "Optuna",
        "Balanceo": best_balance,
        "CV Accuracy": np.nan,
        "CV Precision": np.nan,
        "CV Recall": study.best_value,
        "CV F1": np.nan,
        "CV AUC": np.nan,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0),
        "Test AUC": roc_auc_score(y_test, y_score),
        "Tiempo (s)": elapsed_time,
        "Evaluaciones": n_trials,
        "Best Params": study.best_params,
        "Best Estimator": best_model,
        "Matriz Confusión": confusion_matrix(y_test, y_pred),
        "Predicciones": y_pred,
        "Scores": y_score,
        "Study": study
    }

In [ ]:
!pip install deap
from deap import base, creator, tools
import random

In [ ]:
deap_spaces = {
    "logistic_l2": {
        "C": [0.01, 0.1, 1, 10],
        "balance_strategy": ["none", "smote", "adasyn", "class_weight"]
    },
    "logistic_l1": {
        "C": [0.01, 0.1, 1, 10],
        "balance_strategy": ["none", "smote", "adasyn", "class_weight"]
    },
    "knn": {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"],
        "balance_strategy": ["none", "smote", "adasyn"]
    },
    "naive_bayes": {
        "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6],
        "balance_strategy": ["none", "smote", "adasyn"]
    },
    "decision_tree": {
        "max_depth": [5, 10, 15, None],
        "min_samples_split": [2, 10, 30],
        "min_samples_leaf": [1, 5, 10],
        "balance_strategy": ["none", "smote", "adasyn", "class_weight"]
    },
    "random_forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 20, None],
        "min_samples_leaf": [1, 10],
        "balance_strategy": ["none", "smote", "adasyn", "class_weight"], 
        "max_features": ["sqrt", "log2"] 
    },
    "xgboost": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 20],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 5],
        "balance_strategy": ["none", "smote", "adasyn", "scale_pos_weight"]
    },
    "svm": {
        "C": [0.01, 0.1, 1, 10, 100],
        "gamma": ["scale", 0.001, 0.01, 0.1, 1],
        "balance_strategy": ["none", "smote", "adasyn", "class_weight"]
    }
}

In [ ]:
def individual_to_params(individual, model_key):
    space = deap_spaces[model_key]
    keys = list(space.keys())

    params = {}

    for i, key in enumerate(keys):
        params[key] = space[key][individual[i]]

    balance_strategy = params.pop("balance_strategy")

    return params, balance_strategy

In [ ]:
def evaluate_deap_individual(individual, model_key, scale_numeric=True):

    params, balance_strategy = individual_to_params(individual, model_key)

    X_train_used, y_train_used = get_train_data(model_key)

    fold_scores = []

    for train_idx, val_idx in cv_strategy.split(X_train_used, y_train_used):

        X_fold_train = X_train_used.iloc[train_idx]
        X_fold_val = X_train_used.iloc[val_idx]

        y_fold_train = y_train_used.iloc[train_idx]
        y_fold_val = y_train_used.iloc[val_idx]

        # Preprocesamiento ajustado solo sobre train fold
        preprocessor_fold = build_preprocessor(scale_numeric=scale_numeric)

        X_fold_train_processed = preprocessor_fold.fit_transform(X_fold_train)
        X_fold_val_processed = preprocessor_fold.transform(X_fold_val)

        # Balanceo solo sobre train fold
        if balance_strategy == "smote":
            sampler = SMOTE(random_state=RANDOM_STATE)
            X_fold_train_processed, y_fold_train = sampler.fit_resample(
                X_fold_train_processed,
                y_fold_train
            )

        elif balance_strategy == "adasyn":
            sampler = ADASYN(random_state=RANDOM_STATE)
            X_fold_train_processed, y_fold_train = sampler.fit_resample(
                X_fold_train_processed,
                y_fold_train
            )

        model = make_estimator_from_params(
            model_key=model_key,
            params=params,
            balance_strategy=balance_strategy
        )

        model.fit(X_fold_train_processed, y_fold_train)

        y_val_pred = model.predict(X_fold_val_processed)

        fold_scores.append(
            recall_score(y_fold_val, y_val_pred, zero_division=0)
        )

    return (np.mean(fold_scores),)

In [ ]:
def run_deap_model(
    model_key,
    n_population=10,
    n_generations=5,
    cxpb=0.5,
    mutpb=0.2
):

    model_info = classification_models[model_key]
    space = deap_spaces[model_key]
    keys = list(space.keys())

    if "FitnessMax" not in creator.__dict__:
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))

    if "Individual" not in creator.__dict__:
        creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    for i, key in enumerate(keys):
        toolbox.register(
            f"attr_{i}",
            random.randint,
            0,
            len(space[key]) - 1
        )

    toolbox.register(
        "individual",
        tools.initCycle,
        creator.Individual,
        tuple(getattr(toolbox, f"attr_{i}") for i in range(len(keys))),
        n=1
    )

    toolbox.register(
        "population",
        tools.initRepeat,
        list,
        toolbox.individual
    )

    toolbox.register(
        "evaluate",
        evaluate_deap_individual,
        model_key=model_key,
        scale_numeric=model_info["scale_numeric"]
    )

    toolbox.register("mate", tools.cxTwoPoint)

    def mutate_individual(individual):
        gene_idx = random.randint(0, len(individual) - 1)
        key = keys[gene_idx]
        individual[gene_idx] = random.randint(0, len(space[key]) - 1)
        return (individual,)

    toolbox.register("mutate", mutate_individual)
    toolbox.register("select", tools.selTournament, tournsize=3)

    population = toolbox.population(n=n_population)
    logbook = []

    start_time = time.time()

    for gen in range(n_generations):

        invalid_individuals = [ind for ind in population if not ind.fitness.valid]
        fitnesses = list(map(toolbox.evaluate, invalid_individuals))

        for ind, fit in zip(invalid_individuals, fitnesses):
            ind.fitness.values = fit

        best_ind = tools.selBest(population, 1)[0]

        logbook.append({
            "Generacion": gen,
            "Best Recall": best_ind.fitness.values[0],
            "Best Individual": list(best_ind)
        })

        offspring = toolbox.select(population, len(population))
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < cxpb:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < mutpb:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        population[:] = offspring

    invalid_individuals = [ind for ind in population if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_individuals))

    for ind, fit in zip(invalid_individuals, fitnesses):
        ind.fitness.values = fit

    best_individual = tools.selBest(population, 1)[0]
    best_params, best_balance = individual_to_params(best_individual, model_key)

    elapsed_time = time.time() - start_time

    best_model = make_pipeline_for_search(
        model_key=model_key,
        balance_strategy=best_balance,
        scale_numeric=model_info["scale_numeric"]
    )

    best_model.set_params(**{
        f"model__{k}": v for k, v in best_params.items()
    })

    X_train_used, y_train_used = get_train_data(model_key)

    best_model.fit(X_train_used, y_train_used)

    y_pred = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)

    return {
        "Modelo": model_info["name"],
        "Metodo": "DEAP",
        "Balanceo": best_balance,
        "CV Accuracy": np.nan,
        "CV Precision": np.nan,
        "CV Recall": best_individual.fitness.values[0],
        "CV F1": np.nan,
        "CV AUC": np.nan,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0),
        "Test AUC": roc_auc_score(y_test, y_score),
        "Tiempo (s)": elapsed_time,
        "Evaluaciones": n_population * n_generations,
        "Best Params": best_params,
        "Best Estimator": best_model,
        "Matriz Confusión": confusion_matrix(y_test, y_pred),
        "Predicciones": y_pred,
        "Scores": y_score,
        "Logbook": pd.DataFrame(logbook)
    }

---

In [ ]:
!pip install tqdm

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
import os
import joblib
import pandas as pd

BASE_DIR = r"C:\Users\danih\OneDrive\Documentos\U\6° SEMESTRE\Machine L\Proyecto diabetes - ML"

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints_rf")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
def save_checkpoint(result, method_name, balance_strategy):
    path = os.path.join(
        CHECKPOINT_DIR,
        f"rf_{method_name}_{balance_strategy}.joblib"
    )
    joblib.dump(result, path)
    print(f"Guardado: {path}")


def load_existing_results(method_name):
    results = []
    completed = []

    for balance_strategy in get_balance_options("random_forest"):
        path = os.path.join(
            CHECKPOINT_DIR,
            f"rf_{method_name}_{balance_strategy}.joblib"
        )

        if os.path.exists(path):
            result = joblib.load(path)
            results.append(result)
            completed.append(balance_strategy)

    return results, completed

In [ ]:
optuna_path = f"{CHECKPOINT_DIR}/rf_optuna.joblib"

if os.path.exists(optuna_path):
    optuna_results_rf = [joblib.load(optuna_path)]
    print("Optuna Random Forest ya existe. Se cargó desde checkpoint.")
else:
    optuna_results_rf = []

    result = run_optuna_model(
        model_key="random_forest",
        model_name="Random Forest",
        scale_numeric=False,
        n_trials=30
    )

    optuna_results_rf.append(result)
    joblib.dump(result, optuna_path)
    print(f"Guardado: {optuna_path}")

In [ ]:
# Resumen Random Forest - Optuna

all_results_rf_optuna = optuna_results_rf

rf_optuna_summary = pd.DataFrame([
    {
        "Modelo": r.get("Modelo"),
        "Metodo": r.get("Metodo"),
        "Balanceo": r.get("Balanceo"),

        "CV Accuracy": r.get("CV Accuracy"),
        "CV Precision": r.get("CV Precision"),
        "CV Recall": r.get("CV Recall"),
        "CV F1": r.get("CV F1"),
        "CV AUC": r.get("CV AUC"),

        "Test Accuracy": r.get("Test Accuracy"),
        "Test Precision": r.get("Test Precision"),
        "Test Recall": r.get("Test Recall"),
        "Test F1": r.get("Test F1"),
        "Test AUC": r.get("Test AUC"),

        "Tiempo (s)": r.get("Tiempo (s)"),
        "Evaluaciones": r.get("Evaluaciones"),
        "Best Params": str(r.get("Best Params"))
    }
    for r in all_results_rf_optuna
])

display(
    rf_optuna_summary
    .sort_values(by="Test Recall", ascending=False, na_position="last")
    .round(4)
)

---

In [ ]:

#Guardado completo de resultados del modelo

import os
import joblib
import pandas as pd
import numpy as np

OUTPUT_DIR = "resultados_modelos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cambiar según el modelo trabajado
modelo_nombre = "random_forest_optuna"

# Unir resultados generados para este modelo
all_results_model = all_results_rf_optuna


# Tabla resumen completa

summary_model = pd.DataFrame([
    {
        "Modelo": r.get("Modelo"),
        "Metodo": r.get("Metodo"),
        "Balanceo": r.get("Balanceo"),

        # Métricas de validación cruzada
        "CV Accuracy": r.get("CV Accuracy"),
        "CV Precision": r.get("CV Precision"),
        "CV Recall": r.get("CV Recall"),
        "CV F1": r.get("CV F1"),
        "CV AUC": r.get("CV AUC"),

        # Métricas en test
        "Test Accuracy": r.get("Test Accuracy"),
        "Test Precision": r.get("Test Precision"),
        "Test Recall": r.get("Test Recall"),
        "Test F1": r.get("Test F1"),
        "Test AUC": r.get("Test AUC"),

        # Información computacional
        "Tiempo (s)": r.get("Tiempo (s)"),
        "Evaluaciones": r.get("Evaluaciones"),

        # Configuración encontrada
        "Best Params": str(r.get("Best Params"))
    }
    for r in all_results_model
])

# Ordenar por la métrica principal del proyecto
summary_model = summary_model.sort_values(
    by="Test Recall",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

display(summary_model.round(4))

# ------------------------------------------------------------
# Guardar tabla resumen
# ------------------------------------------------------------

summary_path = f"{OUTPUT_DIR}/summary_{modelo_nombre}.csv"

summary_model.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# Guardar objetos completos
# ------------------------------------------------------------

objects_path = f"{OUTPUT_DIR}/objects_{modelo_nombre}.joblib"

joblib.dump(
    all_results_model,
    objects_path
)

# ------------------------------------------------------------
# Guardar mejor resultado según Test Recall
# ------------------------------------------------------------

valid_results = [
    r for r in all_results_model
    if r.get("Test Recall") is not None
]

if len(valid_results) == 0:
    raise ValueError("No hay resultados válidos con Test Recall para seleccionar el mejor modelo.")

best_result_model = max(
    valid_results,
    key=lambda r: r.get("Test Recall")
)

best_result_path = f"{OUTPUT_DIR}/best_result_{modelo_nombre}.joblib"

joblib.dump(
    best_result_model,
    best_result_path
)

# Si el mejor resultado tiene estimador entrenado, guardarlo aparte
if best_result_model.get("Best Estimator") is not None:
    best_estimator_path = f"{OUTPUT_DIR}/best_estimator_{modelo_nombre}.joblib"

    joblib.dump(
        best_result_model["Best Estimator"],
        best_estimator_path
    )

print("Archivos guardados correctamente:")
print(f"- Tabla resumen: {summary_path}")
print(f"- Objetos completos: {objects_path}")
print(f"- Mejor resultado: {best_result_path}")

if best_result_model.get("Best Estimator") is not None:
    print(f"- Mejor estimador: {best_estimator_path}")